# SWS3009A Cat Detection + Breed Classification

Pipeline:

```text
camera / scene image -> pretrained YOLO detects cat -> crop cat -> EfficientNet-B0 classifies breed
```

Classes: `ragdoll`, `singapura`, `persian`, `sphynx`, `pallas`.

Use this notebook in Google Colab first. After training, download `best_effnet_b0_cat_breeds.pth` and `class_to_idx.json` for local robot inference.

## 1. Install dependencies

Run this once. In Colab, choose `Runtime -> Change runtime type -> GPU` before running training.

In [ ]:
import importlib.util
import subprocess
import sys

def ensure_package(import_name, pip_name=None):
    pip_name = pip_name or import_name
    if importlib.util.find_spec(import_name) is None:
        print(f"Installing {pip_name}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-U", pip_name])

ensure_package("ultralytics")
ensure_package("sklearn", "scikit-learn")
ensure_package("cv2", "opencv-python")
ensure_package("PIL", "pillow")
ensure_package("tqdm")

print("Dependencies ready")

## 2. Imports and configuration

Put your collected images here:

```text
cat_breed_dataset/raw/ragdoll
cat_breed_dataset/raw/singapura
cat_breed_dataset/raw/persian
cat_breed_dataset/raw/sphynx
cat_breed_dataset/raw/pallas
```

Put robot-view test scene images here:

```text
test_scene/
```

In [ ]:
from pathlib import Path
import json
import math
import os
import random
import shutil
import time

import cv2
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
from sklearn.metrics import classification_report, confusion_matrix
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from torchvision.models import efficientnet_b2, EfficientNet_B2_Weights
from tqdm.auto import tqdm
from ultralytics import YOLO

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

CLASS_NAMES = ["ragdoll", "singapura", "persian", "sphynx", "pallas"]
DATA_ROOT = Path("cat_breed_dataset")
RAW_DIR = DATA_ROOT / "raw"
TRAIN_DIR = DATA_ROOT / "train"
VAL_DIR = DATA_ROOT / "val"
TEST_SCENE_DIR = Path("test_scene")

# YOLO ensemble: try larger models if smaller ones miss
YOLO_ENSEMBLE = ["yolo26n.pt", "yolo11s.pt", "yolo11m.pt"]
COCO_CAT_CLASS_ID = 15

BATCH_SIZE = 16
EPOCHS = 15
LR = 1e-4
IMG_SIZE = 260
VAL_RATIO = 0.15

BEST_MODEL_PATH = Path("best_effnet_b2_cat_breeds.pth")
CLASS_MAP_PATH = Path("class_to_idx.json")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

## 3. Create folders

In [ ]:
for base in [RAW_DIR, TRAIN_DIR, VAL_DIR]:
    for cls in CLASS_NAMES:
        (base / cls).mkdir(parents=True, exist_ok=True)
TEST_SCENE_DIR.mkdir(parents=True, exist_ok=True)

print("Folders created.")
print("Put raw images under:", RAW_DIR.resolve())
print("Put YOLO scene test images under:", TEST_SCENE_DIR.resolve())

## 4. Count raw images

Run this after uploading/collecting images.

In [ ]:
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}

def list_images(folder):
    folder = Path(folder)
    if not folder.exists():
        return []
    return sorted([p for p in folder.rglob("*") if p.suffix.lower() in IMAGE_EXTS])

def count_by_class(root):
    root = Path(root)
    counts = {}
    for cls in CLASS_NAMES:
        counts[cls] = len(list_images(root / cls))
    counts["total"] = sum(counts.values())
    return counts

print("Raw counts:", count_by_class(RAW_DIR))

## 5. Split raw images into train/val

Default is non-destructive. If you want to redo the split, set `RESET_EXISTING_SPLIT = True`.

In [ ]:
RESET_EXISTING_SPLIT = False

def folder_has_images(root):
    return any(len(list_images(Path(root) / cls)) > 0 for cls in CLASS_NAMES)

def split_dataset(raw_dir=RAW_DIR, train_dir=TRAIN_DIR, val_dir=VAL_DIR, val_ratio=VAL_RATIO, reset=False):
    raw_dir = Path(raw_dir)
    train_dir = Path(train_dir)
    val_dir = Path(val_dir)

    if reset:
        for target in [train_dir, val_dir]:
            if target.exists():
                shutil.rmtree(target)
            for cls in CLASS_NAMES:
                (target / cls).mkdir(parents=True, exist_ok=True)

    if not reset and (folder_has_images(train_dir) or folder_has_images(val_dir)):
        print("train/val already contain images. Set RESET_EXISTING_SPLIT=True to recreate them.")
        return

    for cls in CLASS_NAMES:
        src_files = list_images(raw_dir / cls)
        random.shuffle(src_files)
        n_val = max(1, int(round(len(src_files) * val_ratio))) if src_files else 0
        val_files = src_files[:n_val]
        train_files = src_files[n_val:]

        (train_dir / cls).mkdir(parents=True, exist_ok=True)
        (val_dir / cls).mkdir(parents=True, exist_ok=True)

        for p in train_files:
            shutil.copy2(p, train_dir / cls / p.name)
        for p in val_files:
            shutil.copy2(p, val_dir / cls / p.name)

        print(f"{cls}: train={len(train_files)}, val={len(val_files)}")

split_dataset(reset=RESET_EXISTING_SPLIT)
print("Train counts:", count_by_class(TRAIN_DIR))
print("Val counts:", count_by_class(VAL_DIR))

## 6. Test pretrained YOLO on robot-view scene images

This checks whether pretrained YOLO can detect the cat in large scene images. If this fails often, let the robot move closer and use CNN classification directly, or collect scene images and fine-tune YOLO later.

In [ ]:
def load_yolo_ensemble(candidates=YOLO_MODEL_CANDIDATES):
    models = []
    for model_name in candidates:
        try:
            print("Loading YOLO model:", model_name)
            model = YOLO(model_name)
            models.append((model, model_name))
            print("Loaded:", model_name)
        except Exception as exc:
            print("Failed:", model_name, exc)
    if not models:
        raise RuntimeError(f"Could not load any YOLO model")
    return models

detectors = load_yolo_ensemble(["yolo26n.pt", "yolo11s.pt", "yolo11m.pt"])

scene_images = list_images(TEST_SCENE_DIR)
print("Scene images:", len(scene_images))

if scene_images:
    # Cascade: try each model until one detects cats
    for detector, mname in detectors:
        yolo_results = detector.predict(
            source=str(TEST_SCENE_DIR),
            classes=[COCO_CAT_CLASS_ID],
            conf=0.25,
            imgsz=640,
            save=True
        )
        if any(len(r.boxes) > 0 for r in yolo_results):
            print(f"Detection by: {mname}")
            break
    print("YOLO outputs saved under runs/detect/...")
else:
    print("No scene images yet. Add images to test_scene/ and rerun this cell.")

## 7. Build and train EfficientNet-B0 classifier

In [ ]:
def build_classifier(num_classes=5, pretrained=True):
    weights = EfficientNet_B2_Weights.DEFAULT if pretrained else None
    model = efficientnet_b2(weights=weights)
    in_features = model.classifier[1].in_features
    model.classifier[1] = nn.Linear(in_features, num_classes)
    return model

train_tfms = transforms.Compose([
    transforms.Resize((288, 288)),
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.75, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

val_tfms = transforms.Compose([
    transforms.Resize((288, 288)),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

train_ds = datasets.ImageFolder(TRAIN_DIR, transform=train_tfms)
val_ds = datasets.ImageFolder(VAL_DIR, transform=val_tfms)

if len(train_ds) == 0 or len(val_ds) == 0:
    raise RuntimeError("train/val dataset is empty. Upload images and run the split cell first.")

print("class_to_idx:", train_ds.class_to_idx)
print("Train images:", len(train_ds), "Val images:", len(val_ds))

num_workers = 2 if os.name != "nt" else 0
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=num_workers)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=num_workers)

model = build_classifier(num_classes=len(train_ds.classes), pretrained=True).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
best_val_acc = 0.0
best_epoch = 0

for epoch in range(1, EPOCHS + 1):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in tqdm(train_loader, desc=f"Epoch {epoch}/{EPOCHS} train"):
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * labels.size(0)
        preds = outputs.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    train_loss = running_loss / total
    train_acc = correct / total

    model.eval()
    val_loss_sum = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():
        for images, labels in tqdm(val_loader, desc=f"Epoch {epoch}/{EPOCHS} val"):
            images = images.to(device)
            labels = labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            val_loss_sum += loss.item() * labels.size(0)
            preds = outputs.argmax(dim=1)
            val_correct += (preds == labels).sum().item()
            val_total += labels.size(0)

    val_loss = val_loss_sum / val_total
    val_acc = val_correct / val_total
    scheduler.step()

    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)

    print(f"Epoch {epoch}: train_loss={train_loss:.4f}, train_acc={train_acc:.4f}, val_loss={val_loss:.4f}, val_acc={val_acc:.4f}")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_epoch = epoch
        checkpoint = {
            "arch": "efficientnet_b2",
            "model_state": model.state_dict(),
            "class_to_idx": train_ds.class_to_idx,
            "classes": train_ds.classes,
            "img_size": IMG_SIZE,
            "best_val_acc": best_val_acc,
            "epoch": epoch,
        }
        torch.save(checkpoint, BEST_MODEL_PATH)
        with open(CLASS_MAP_PATH, "w") as f:
            json.dump(train_ds.class_to_idx, f, indent=2)
        print("Saved new best model:", BEST_MODEL_PATH)

print("Best val_acc:", best_val_acc, "at epoch", best_epoch)
print("Final train_acc:", history["train_acc"][-1])
print("Use these in the answer book:")
print("Training Accuracy (%):", round(history["train_acc"][-1] * 100, 2))
print("Validation Accuracy (%):", round(best_val_acc * 100, 2))

## 8. Plot training curves

In [ ]:
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.plot(history["train_acc"], label="train_acc")
plt.plot(history["val_acc"], label="val_acc")
plt.xlabel("epoch")
plt.ylabel("accuracy")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history["train_loss"], label="train_loss")
plt.plot(history["val_loss"], label="val_loss")
plt.xlabel("epoch")
plt.ylabel("loss")
plt.legend()
plt.tight_layout()
plt.show()

## 9. Validation classification report

In [ ]:
checkpoint = torch.load(BEST_MODEL_PATH, map_location=device)
eval_model = build_classifier(num_classes=len(checkpoint["classes"]), pretrained=False).to(device)
eval_model.load_state_dict(checkpoint["model_state"])
eval_model.eval()

all_preds = []
all_labels = []
with torch.no_grad():
    for images, labels in val_loader:
        images = images.to(device)
        outputs = eval_model(images)
        preds = outputs.argmax(dim=1).cpu().numpy().tolist()
        all_preds.extend(preds)
        all_labels.extend(labels.numpy().tolist())

print(classification_report(all_labels, all_preds, target_names=checkpoint["classes"]))
print("Confusion matrix:")
print(confusion_matrix(all_labels, all_preds))

## 10. End-to-end test on one scene image

This loads YOLO, finds the cat, crops it, then classifies breed with the trained CNN.

In [ ]:
def load_trained_classifier(model_path=BEST_MODEL_PATH):
    checkpoint = torch.load(model_path, map_location=device)
    classes = checkpoint.get("classes")
    if classes is None:
        class_to_idx = checkpoint["class_to_idx"]
        classes = [None] * len(class_to_idx)
        for name, idx in class_to_idx.items():
            classes[idx] = name
    model = build_classifier(num_classes=len(classes), pretrained=False).to(device)
    state = checkpoint.get("model_state", checkpoint)
    model.load_state_dict(state)
    model.eval()
    return model, classes

infer_tfms = transforms.Compose([
    transforms.Resize((288, 288)),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

def classify_crop(crop_bgr, classifier, class_names, min_conf=0.6):
    if crop_bgr is None or crop_bgr.shape[0] < 20 or crop_bgr.shape[1] < 20:
        return None, 0.0
    try:
        crop_rgb = cv2.cvtColor(crop_bgr, cv2.COLOR_BGR2RGB)
        pil_img = Image.fromarray(crop_rgb)
        tensor = infer_tfms(pil_img).unsqueeze(0).to(device)
        with torch.no_grad():
            logits = classifier(tensor)
            probs = torch.softmax(logits, dim=1)[0]
            pred_idx = int(probs.argmax().item())
            conf = float(probs[pred_idx].item())
        return class_names[pred_idx], conf
    except Exception:
        return None, 0.0

def crop_with_margin(frame, xyxy, margin=0.08):
    h, w = frame.shape[:2]
    x1, y1, x2, y2 = [int(v) for v in xyxy]
    bw = x2 - x1
    bh = y2 - y1
    dx = int(bw * margin)
    dy = int(bh * margin)
    x1 = max(0, x1 - dx)
    y1 = max(0, y1 - dy)
    x2 = min(w, x2 + dx)
    y2 = min(h, y2 + dy)
    return frame[y1:y2, x1:x2], (x1, y1, x2, y2)

def predict_scene_image(image_path, output_path="pipeline_result.jpg", conf=0.25):
    classifier, classes = load_trained_classifier()
    frame = cv2.imread(str(image_path))
    if frame is None:
        raise FileNotFoundError(f"Could not read image: {image_path}")

    pred = None

    # ── Cascade YOLO ensemble: try each model until one detects ──
    for detector, mname in detectors:
        results = detector.predict(frame, classes=[COCO_CAT_CLASS_ID], conf=conf, imgsz=640, verbose=False)
        if len(results[0].boxes) > 0:
            boxes = results[0].boxes
            best_i = int(boxes.conf.argmax().item())
            xyxy = boxes.xyxy[best_i].cpu().numpy()
            det_conf = float(boxes.conf[best_i].item())
            crop, (x1, y1, x2, y2) = crop_with_margin(frame, xyxy)
            breed, cls_conf = classify_crop(crop, classifier, classes, min_conf=0.0)
            if breed:
                pred = {"breed": breed, "classification_confidence": cls_conf, "detection_confidence": det_conf, "method": mname, "box": (x1, y1, x2, y2)}
            break

    # ── Result ──
    if pred:
        x1, y1, x2, y2 = pred["box"]
        cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
        label = f"{pred['breed']} {pred['method']} cls={pred['classification_confidence']:.2f} det={pred['detection_confidence']:.2f}"
        cv2.putText(frame, label, (x1, max(25, y1 - 10)), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
        cv2.imwrite(output_path, frame)
        print("Prediction:", pred["breed"])
        print("Method:", pred["method"])
        print("Classification confidence:", round(pred["classification_confidence"], 4))
    else:
        cv2.imwrite(output_path, frame)
        print("No cat detected by any model")

    print("Saved:", output_path)
    return pred

scene_images = list_images(TEST_SCENE_DIR)
if scene_images:
    predict_scene_image(scene_images[0], output_path="pipeline_result.jpg", conf=0.25)
else:
    print("Add a scene image to test_scene/ first.")

## 11. Files to download after training

Download these two files and put them in `outputs/classifier/` next to the local robot inference script:

```text
outputs/classifier/best_effnet_b2_cat_breeds.pth
outputs/classifier/class_to_idx.json
```

For the report, use:

```text
Training Accuracy = final train_acc * 100
Validation Accuracy = best val_acc * 100
```